# 08 · 轨迹预测：多模态候选与 ADE/FDE/Miss Rate

预测模块不是“猜一条平均轨迹”。当道路允许直行、左转或右转时，合理系统需要表达多个未来模式，并用 minADE、minFDE、miss rate 等指标评估候选集合。

本 notebook 不训练大型预测网络，而是构造一个可解释的 multi-modal baseline，重点练习：

- trajectory tensor 的形状和坐标约定；
- ADE、FDE、minADE、minFDE 和 miss rate；
- K 个候选模式如何覆盖真实未来；
- open-loop metric 为什么不能直接等价于闭环驾驶安全。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

rng = np.random.default_rng(12)
horizon = 30
dt = 0.2
future_time = np.arange(1, horizon + 1) * dt
speed = 6.0

def trajectory(mode, noise=0.0, seed=0):
    local = np.random.default_rng(seed)
    x = speed * future_time
    if mode == 'straight':
        y = np.zeros_like(x)
    elif mode == 'left':
        y = 2.8 * (future_time / future_time[-1]) ** 1.6
    else:
        y = -2.8 * (future_time / future_time[-1]) ** 1.6
    result = np.c_[x, y]
    return result + local.normal(0, noise, size=result.shape)

def build_batch(n=240, prediction_noise=0.25, seed=12):
    local = np.random.default_rng(seed)
    modes = local.choice(['straight', 'left', 'right'], size=n, p=[0.5, 0.25, 0.25])
    truth = np.stack([trajectory(mode, noise=0.08, seed=seed + i) for i, mode in enumerate(modes)])
    mode_set = ['straight', 'left', 'right']
    predictions = np.stack([
        np.stack([trajectory(candidate, noise=prediction_noise, seed=seed + 1000 + i * 10 + k) for k in range(n)])
        for i, candidate in enumerate(mode_set)
    ], axis=1)
    return modes, truth, predictions

modes, truth, predictions = build_batch()
print('truth shape:', truth.shape, 'prediction shape:', predictions.shape)


In [ ]:
def prediction_metrics(truth, predictions, miss_threshold=2.0):
    errors = np.linalg.norm(predictions - truth[:, None, :, :], axis=-1)
    ade = errors.mean(axis=-1)
    fde = errors[:, :, -1]
    minade = ade.min(axis=1)
    minfde = fde.min(axis=1)
    miss_rate = np.mean(minfde > miss_threshold)
    return {
        'ADE_by_mode': ade.mean(axis=0),
        'FDE_by_mode': fde.mean(axis=0),
        'minADE': minade.mean(),
        'minFDE': minfde.mean(),
        'miss_rate': miss_rate,
        'errors': errors,
    }

metrics = prediction_metrics(truth, predictions)
print({key:value for key,value in metrics.items() if key != 'errors'})


In [ ]:
def show_prediction(prediction_noise=0.25, miss_threshold=2.0, sample_index=3):
    modes, truth, predictions = build_batch(prediction_noise=prediction_noise)
    metrics = prediction_metrics(truth, predictions, miss_threshold)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for k, label in enumerate(['straight', 'left', 'right']):
        ax[0].plot(predictions[sample_index, k, :, 0], predictions[sample_index, k, :, 1], label=label)
    ax[0].plot(truth[sample_index, :, 0], truth[sample_index, :, 1], color='black', linewidth=2, label=f'truth={modes[sample_index]}')
    ax[0].set_title('multi-modal hypotheses')
    ax[0].set_xlabel('x / m')
    ax[0].set_ylabel('y / m')
    ax[0].legend()
    errors = metrics['errors']
    ax[1].hist(errors[:, :, -1].min(axis=1), bins=20)
    ax[1].axvline(miss_threshold, color='tab:red', linestyle='--')
    ax[1].set_title(f"minFDE={metrics['minFDE']:.3f}, miss={metrics['miss_rate']:.3f}")
    ax[1].set_xlabel('best final displacement error / m')
    plt.tight_layout()
    plt.show()

interact(
    show_prediction,
    prediction_noise=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.25, description='prediction noise'),
    miss_threshold=FloatSlider(min=0.5, max=4.0, step=0.25, value=2.0, description='miss threshold'),
    sample_index=IntSlider(min=0, max=20, step=1, value=3, description='sample'),
);


### 练习：指标与系统目标

- 比较 K=1 的平均轨迹与 K=3 的多模态候选，解释为什么平均轨迹可能落在不可行区域。
- 扫描 miss threshold，画 miss rate 曲线。
- 给左转/右转错误加入更高代价，构造一个 route-conditioned metric。
- 想一想：预测 ADE 很低但与 ego 规划轨迹碰撞时，系统应该如何报告？


In [ ]:
thresholds = np.linspace(0.5, 4.0, 15)
miss_rates = [prediction_metrics(truth, predictions, value)['miss_rate'] for value in thresholds]
plt.plot(thresholds, miss_rates, marker='o')
plt.xlabel('miss threshold / m')
plt.ylabel('miss rate')
plt.title('miss-rate sensitivity')
plt.show()


## 完成标准

提交一份小报告：

1. 给出 ADE、FDE、minADE、minFDE、miss rate。
2. 展示一个多模态覆盖成功的案例和一个 mode collapse / miss 案例。
3. 区分 open-loop trajectory error、collision risk 和闭环 performance。
4. 说明下一步如何把 agent interaction、map lane topology 和 uncertainty 加入预测模型。
